In [ ]:
import sys

!{sys.executable} -m pip install -q medmnist tensorboardX
!{sys.executable} -m pip install -q acsconv
!{sys.executable} -m pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import os
import time
from copy import deepcopy

import medmnist
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data

from torchvision.models import resnet50, ResNet50_Weights
from medmnist import INFO, Evaluator
from tqdm import trange

from acsconv.converters import ACSConverter

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
data_flag = 'organmnist3d'
download = True

NUM_EPOCHS = 100
BATCH_SIZE = 16
lr = 0.0003
SIZE = 28

In [ ]:
info = INFO[data_flag]

task = info['task']
n_classes = len(info['label'])

DataClass = getattr(medmnist, info['python_class'])

print("Task:", task)
print("Number of classes:", n_classes)

In [ ]:
train_dataset = DataClass(
    split='train',
    download=download,
    size=SIZE,
    as_rgb=True
)

val_dataset = DataClass(
    split='val',
    download=download,
    size=SIZE,
    as_rgb=True
)

test_dataset = DataClass(
    split='test',
    download=download,
    size=SIZE,
    as_rgb=True
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
def cache_to_gpu(dataset):
    inputs = torch.stack([
        torch.tensor(dataset[i][0]).float()
        for i in range(len(dataset))
    ]).to(device)
    targets = torch.tensor([
        int(dataset[i][1].item())
        for i in range(len(dataset))
    ]).long().to(device)
    return inputs, targets

print("Caching datasets to GPU...")
train_inputs, train_targets = cache_to_gpu(train_dataset)
val_inputs, val_targets = cache_to_gpu(val_dataset)
test_inputs, test_targets = cache_to_gpu(test_dataset)
print(f"Done — train: {train_inputs.shape}, val: {val_inputs.shape}, test: {test_inputs.shape}")

In [ ]:
train_loader = data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=8,
    pin_memory=True
)

val_loader = data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=8,
    pin_memory=True
)

test_loader = data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=8,
    pin_memory=True
)

In [ ]:
model = resnet50(weights=ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, n_classes)

model = ACSConverter(model)
model = model.to(device)

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=lr,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[50, 75],
    gamma=0.1
)

In [ ]:
train_evaluator = Evaluator(data_flag, 'train', size=SIZE)
val_evaluator = Evaluator(data_flag, 'val', size=SIZE)
test_evaluator = Evaluator(data_flag, 'test', size=SIZE)

In [ ]:
def train_epoch(cached_inputs, cached_targets):

    model.train()
    total_loss = []

    perm = torch.randperm(len(cached_inputs), device=device)
    inputs_shuffled = cached_inputs[perm]
    targets_shuffled = cached_targets[perm]

    for i in range(0, len(inputs_shuffled), BATCH_SIZE):
        inputs = inputs_shuffled[i:i + BATCH_SIZE]
        targets = targets_shuffled[i:i + BATCH_SIZE]

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        total_loss.append(loss.item())

    return np.mean(total_loss)


In [ ]:
def evaluate(cached_inputs, cached_targets, evaluator):

    model.eval()
    total_loss = []
    y_score = torch.tensor([]).to(device)

    with torch.no_grad():

        for i in range(0, len(cached_inputs), BATCH_SIZE):
            inputs = cached_inputs[i:i + BATCH_SIZE]
            targets = cached_targets[i:i + BATCH_SIZE]

            outputs = model(inputs)

            loss = criterion(outputs, targets)

            outputs = torch.softmax(outputs, dim=1)

            total_loss.append(loss.item())

            y_score = torch.cat((y_score, outputs), 0)

    y_score = y_score.cpu().numpy()

    auc, acc = evaluator.evaluate(y_score)

    return np.mean(total_loss), auc, acc


In [ ]:
best_val_auc = 0

best_test_auc = 0
best_test_acc = 0

best_model_test_auc = 0
best_model_test_acc = 0

for epoch in trange(NUM_EPOCHS):

    train_loss = train_epoch(train_inputs, train_targets)

    val_loss, val_auc, val_acc = evaluate(
        val_inputs,
        val_targets,
        val_evaluator
    )

    test_loss, test_auc, test_acc = evaluate(
        test_inputs,
        test_targets,
        test_evaluator
    )

    scheduler.step()

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")

    print(f"Validation AUC: {val_auc:.4f}")
    print(f"Validation ACC: {val_acc:.4f}")

    print(f"Test AUC: {test_auc:.4f}")
    print(f"Test ACC: {test_acc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_model_test_auc = test_auc
        best_model_test_acc = test_acc

        torch.save(model.state_dict(), "best_organmnist3d_resnet50.pth")
        print("Best model saved!")

    if test_auc > best_test_auc:
        best_test_auc = test_auc

    if test_acc > best_test_acc:
        best_test_acc = test_acc

In [ ]:
print("\nTraining complete!")

print("\n===== FINAL RESULTS =====")
print(f"Best Validation AUC (saved model): {best_val_auc:.4f}")
print(f"Saved model Test AUC: {best_model_test_auc:.4f}")
print(f"Saved model Test ACC: {best_model_test_acc:.4f}")

print(f"\nHighest Test AUC observed: {best_test_auc:.4f}")
print(f"Highest Test ACC observed: {best_test_acc:.4f}")